# CatBoost регрессия

Один эксперимент без сохранения результатов.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_percentage_error
from catboost import CatBoostRegressor


def cv_cat(model, X, y, cat_idx):
    kf = KFold(n_splits=5)
    scores = []
    for tr, va in kf.split(X, y):
        model.fit(X.iloc[tr], y.iloc[tr], cat_features=cat_idx, verbose=False)
        pred = model.predict(X.iloc[va])
        scores.append(mean_absolute_percentage_error(y.iloc[va], pred))
    print(f"MAPE: {np.mean(scores):.6f} +- {np.std(scores):.6f}")
    return float(np.mean(scores))


df_train = pd.read_csv('data/train.csv')
X = df_train.iloc[:, :-1].copy()
y = df_train.iloc[:, -1].copy()

X[['unified_address_city', 'unified_address_region']] = X[['unified_address_city', 'unified_address_region']].fillna('missing')
for c in ['key_skills_name', 'languages_name', 'employer_industries']:
    if c in X.columns:
        X[c] = X[c].fillna('missing')

X = X.drop(columns=['id','employer_id','raw_description','raw_branded_description','lemmaized_wo_stopwords_raw_description','lemmaized_wo_stopwords_raw_branded_description','name','unified_address_country'], errors='ignore').copy()

counts = X['employer_name'].value_counts()
rare_categories = counts[counts < 200].index
X['employer_name'] = X['employer_name'].replace(rare_categories, 'Other')

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
for c in cat_cols:
    X[c] = X[c].fillna('missing').astype(str)
cat_idx = [X.columns.get_loc(c) for c in cat_cols]


## Эксперимент 

In [2]:
model = CatBoostRegressor(
    loss_function='MAPE',
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    verbose=False,
)

res = cv_cat(model, X, y, cat_idx)
res


MAPE: 0.370778 +- 0.004451


0.3707779580164072